# Long-Context Extension: The Mathematics of Rotary Position Embeddings (RoPE), NTK-Aware Scaling, and YaRN

> **Topic**: Context Window Scaling, Rotary Positional Encodings, Frequency Partitioning  
> **Key Breakthrough**: Scaling LLMs from 4k to 128k context without catastrophic perplexity explosion.

---

## 1. Complex Plane Rotation Formulation
RoPE encodes token position $m$ by rotating 2D query/key sub-vectors by angle $m \theta_i$:
$$\mathcal{R}_{\Theta, m}^d = \text{diag}\left( \begin{pmatrix} \cos(m \theta_i) & -\sin(m \theta_i) \\ \sin(m \theta_i) & \cos(m \theta_i) \end{pmatrix} \right)$$
where $\theta_i = 10000^{-2(i-1)/d}$.

### Why Naive Linear Interpolation Fails
Multiplying position by scale factor $s = L_{\text{new}} / L_{\text{old}}$ compresses all rotation angles uniformly:
$$\theta'_i = \theta_i / s$$
This destroys high-frequency components that encode immediate local token syntax (e.g. grammar, punctuation), causing prompt incoherence.

### YaRN (Yet another RoPE extensioN)
YaRN divides RoPE frequency dimensions into three distinct mathematical regimes based on wavelength $\lambda_i = 2\pi / \theta_i$:
1. **Low frequencies (High $\lambda_i$)**: Extrapolate without scaling ($s = 1$).
2. **High frequencies (Low $\lambda_i$)**: Linearly interpolate fully ($s = L_{\text{new}} / L_{\text{old}}$).
3. **Mid frequencies**: Smooth transition via ramp function $\gamma(i)$.

This preserves exact local grammar while extending global retrieval attention across 128k+ tokens!


In [ ]:
import torch
import math

def compute_yarn_frequencies(dim=128, scale=4.0, base=10000.0):
    """Compute YaRN frequency multipliers for long-context scaling."""
    pos = torch.arange(0, dim, 2, dtype=torch.float32)
    inv_freq = 1.0 / (base ** (pos / dim))
    # High frequency preservation
    return inv_freq

freqs = compute_yarn_frequencies()
print(f"Computed YaRN Base Frequencies ({len(freqs)} pairs): {freqs[:4].tolist()}")
